In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/fraud-project/'
print("Drive connected!")

Mounted at /content/drive
Drive connected!


In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio -q
print("Done!")

Done!


In [3]:
import shutil

shutil.copy(DRIVE_PATH + 'best_model.pkl',     '/content/best_model.pkl')
shutil.copy(DRIVE_PATH + 'processed_data.pkl', '/content/processed_data.pkl')

print("Files copied!")

Files copied!


In [4]:
#  Check Feature Count
import pickle

with open('/content/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

FEATURE_COLS = list(data['X_test'].columns)
print(f"Total features: {len(FEATURE_COLS)}")
print(f"First 10: {FEATURE_COLS[:10]}")

Total features: 225
First 10: ['TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1']


In [5]:
# app.py
app_code = f"""import pickle
import numpy as np
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Fraud Detection API")

with open("/content/best_model.pkl", "rb") as f:
    best = pickle.load(f)

with open("/content/processed_data.pkl", "rb") as f:
    data = pickle.load(f)

model        = best["model"]
THRESHOLD    = best["threshold"]
FEATURE_COLS = list(data["X_test"].columns)

class Transaction(BaseModel):
    TransactionAmt: float = 100.0
    ProductCD:      int   = 0
    card1:          int   = 5000
    hour:           int   = 12

@app.get("/")
def root():
    return {{
        "message":  "Fraud Detection API",
        "status":   "running",
        "features": len(FEATURE_COLS),
        "model":    "XGBoost"
    }}

@app.get("/health")
def health():
    return {{"status": "healthy"}}

@app.post("/predict")
def predict(txn: Transaction):
    row = pd.DataFrame(
        [np.zeros(len(FEATURE_COLS))],
        columns=FEATURE_COLS
    )
    for col, val in [
        ("TransactionAmt", txn.TransactionAmt),
        ("ProductCD",      txn.ProductCD),
        ("card1",          txn.card1),
        ("hour",           txn.hour),
        ("amt_log",        np.log1p(txn.TransactionAmt)),
        ("is_round",       int(txn.TransactionAmt % 1 == 0)),
        ("is_night",       int(txn.hour <= 6)),
    ]:
        if col in FEATURE_COLS:
            row[col] = val

    proba    = model.predict_proba(row)[0][1]
    is_fraud = int(proba >= THRESHOLD)

    return {{
        "is_fraud":   is_fraud,
        "confidence": round(float(proba), 4),
        "threshold":  THRESHOLD,
        "risk_level": "HIGH" if proba > 0.7 else "MEDIUM" if proba > 0.4 else "LOW"
    }}
"""

with open('/content/app.py', 'w') as f:
    f.write(app_code)

# Verify
with open('/content/app.py', 'r') as f:
    saved = f.read()

print(f"Saved: {len(saved)} chars")
print(f"FEATURE_COLS present: {'FEATURE_COLS' in saved}")

Saved: 1712 chars
FEATURE_COLS present: True


In [6]:
#  Launch API
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading
import time

nest_asyncio.apply()

# Kill any existing processes
import subprocess
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['pkill', '-f', 'ngrok'],   capture_output=True)
time.sleep(2)

# Start fresh
ngrok.set_auth_token("3ENVwVwp8OYWONarfMKsuSfBHpK_5VKxuNei4HKh87xE7UiZr")
public_url = ngrok.connect(8000)

BASE_URL = public_url.public_url
print(f"API URL : {BASE_URL}")
print(f"Docs URL: {BASE_URL}/docs")

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, app_dir="/content")

t = threading.Thread(target=run)
t.daemon = True
t.start()
time.sleep(4)
print("\nAPI is running!")

API URL : https://puppy-plenty-bartender.ngrok-free.dev
Docs URL: https://puppy-plenty-bartender.ngrok-free.dev/docs

API is running!


In [7]:
#  Test Root
import requests

r = requests.get(BASE_URL + "/")
print("Root:", r.json())

INFO:     35.194.214.80:0 - "GET / HTTP/1.1" 200 OK
Root: {'message': 'Fraud Detection API', 'status': 'running', 'features': 225, 'model': 'XGBoost'}


In [8]:
#  Test Predict
# Test 1 — Normal transaction
normal = {
    "TransactionAmt": 50.0,
    "ProductCD":      0,
    "card1":          1234,
    "hour":           14
}

r1 = requests.post(BASE_URL + "/predict", json=normal)
print("Normal transaction:", r1.json())

# Test 2 — Suspicious transaction
suspicious = {
    "TransactionAmt": 2000.0,
    "ProductCD":      2,
    "card1":          9999,
    "hour":           3
}

r2 = requests.post(BASE_URL + "/predict", json=suspicious)
print("Suspicious transaction:", r2.json())

INFO:     35.194.214.80:0 - "POST /predict HTTP/1.1" 200 OK
Normal transaction: {'is_fraud': 0, 'confidence': 0.3723, 'threshold': 0.4, 'risk_level': 'LOW'}
INFO:     35.194.214.80:0 - "POST /predict HTTP/1.1" 200 OK
Suspicious transaction: {'is_fraud': 0, 'confidence': 0.2715, 'threshold': 0.4, 'risk_level': 'LOW'}
